In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Read data

In [2]:
with open('../input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print(f'Characters in text: {len(text)}')

Characters in text: 1115394


In [4]:
print(f'{text[:100]}')

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


# Getting unique characters and vocabulary size

In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("".join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


# Character level tokenizer

In [6]:
encoded_dict = {chars[i]: i for i in range(vocab_size)}
decoded_dict = {i: chars[i] for i in range(vocab_size)}

encode = lambda s: [encoded_dict[c] for c in s]
decode = lambda l: "".join([decoded_dict[i] for i in l])

print(encode("di si kompa"))
print(decode(encode("di si kompa")))

[42, 47, 1, 57, 47, 1, 49, 53, 51, 54, 39]
di si kompa


In [7]:
import torch

data = encode(text)
data = torch.tensor(data, dtype=torch.long)
print(data.shape)

torch.Size([1115394])


# Splitting train and val

In [8]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [9]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [10]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f'when input is {context}, target is {target}')

when input is tensor([18]), target is 47
when input is tensor([18, 47]), target is 56
when input is tensor([18, 47, 56]), target is 57
when input is tensor([18, 47, 56, 57]), target is 58
when input is tensor([18, 47, 56, 57, 58]), target is 1
when input is tensor([18, 47, 56, 57, 58,  1]), target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]), target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]), target is 58


In [11]:
torch.manual_seed(1337)

batch_size = 4
block_size = 8

def get_batch(split: str, batch_size):
    data = train_data if split == "train" else val_data
    idx = torch.randint(len(data) - block_size, size=(batch_size, ))
    x = torch.stack([data[i:i+block_size] for i in idx])
    y = torch.stack([data[i+1:i+block_size+1] for i in idx])
    return x, y

xb, yb = get_batch("train", batch_size)
print(xb.shape)
print(xb, "\n--------------------------------------------")
print(yb.shape)
print(yb, "\n--------------------------------------------")

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f'When context is {context}, target is {target}')

torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]]) 
--------------------------------------------
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]]) 
--------------------------------------------
When context is tensor([24]), target is 43
When context is tensor([24, 43]), target is 58
When context is tensor([24, 43, 58]), target is 5
When context is tensor([24, 43, 58,  5]), target is 57
When context is tensor([24, 43, 58,  5, 57]), target is 1
When context is tensor([24, 43, 58,  5, 57,  1]), target is 46
When context is tensor([24, 43, 58,  5, 57,  1, 46]), target is 43
When context is tensor([24, 43, 58,  5, 57,  1, 46, 43]), target is 39
When context is tensor([44]), target is 53
When context is tensor([44, 53]), t

# Simples language model, Bigram

In [12]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token reads directly off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B, T) tensor of integers
        logits = self.token_embedding_table(idx) # (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.reshape(B*T, C)
            targets = targets.reshape(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        
        for _ in range(max_new_tokens):
            # idx is (B,T) array of indices in the current context
            logits, loss = self(idx)
            # focus only on last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)

        return idx

   
model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print(logits.shape)
print(loss)

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)


In [13]:
print(decode(model.generate(idx=torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


Sr?qP-QWktXoL&jLDJgOLVz'RIoDqHdhsV&vLLxatjscMpwLERSPyao.qfzs$Ys$zF-w,;eEkzxjgCKFChs!iWW.ObzDnxA Ms$3


In [14]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

In [15]:
batch_size = 64

for steps in range(1000):

    xb, yb = get_batch("train", batch_size)

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.362992763519287


In [16]:
print(decode(model.generate(idx=torch.zeros((1,1), dtype=torch.long), max_new_tokens=500)[0].tolist()))



Mshe r che RK:
Hofit Pore tomars.
t youtoumatillingleneabe de.

LAu.
Fin IAngom?
The hey! awon, ol colllispred s, whep s wispecLb be ppy!
Bu, bavend asBed

POKIIJor:Q!
KIZAMUDololfo uGAmy!


MINCour, mutaly, aisid,RYeth w.
Th, beait l t WAUCERERDu oway;
Moramps h m wid hes wndofisitingh men s:
I IUCin he ho yock hir sothimow, che y bt.lithieare alororeage:

O, d y, j,
Tonge on hio FonaryougheVeN hicouked,

OKEDUS:
NTy, thicele icars
Hen ghan:
ASie wORORYes is g ane'd cheted
OKeeany llila!
COndo


# Mathematical trick in self attention

In [17]:
torch.manual_seed(1337)

B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

# Basic example of averaging using for loops

- v1

In [18]:
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # up until t-th token, (t, C)
        xbow[b, t] = torch.mean(xprev, dim=0)

xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

# Better averaging using triangular matrix and torch sum

- v2

In [19]:
w = torch.tril(torch.ones((T, T)))
w = w / torch.sum(w, dim=1, keepdim=True)

xbow2 = w @ x

xbow2[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

# Adding softmax to triangular matrix

- v3

In [20]:
tril = torch.tril(torch.ones((T, T)))
w = torch.zeros((T, T))
w = w.masked_fill(tril == 0, float("-inf"))
w = F.softmax(w, dim=-1)
xbow3 = w @ x

xbow3[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

# Self attention

In [21]:
torch.manual_seed(1337)

B, T, C = 4, 8, 32
# random input
x = torch.randn(B, T, C)
x.shape

# single head self attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x) # produces (B, T, head_size)
q = query(x) # produces (B, T, head_size)
v = value(x) # produces (B, T, head_size)

# weight is dot product of q and k
w = q @ k.transpose(-2, -1) / np.sqrt(head_size) # (B, T, head_size) @ (B, head_size, T) = (B, T, T)

# lower triangular matrix of ones
tril = torch.tril(torch.ones((T, T)))
# cover the 0 elements in upper right with -inf, these tokens do not communicate
w = w.masked_fill(tril == 0, float("-inf"))
# apply softmax to normalize
w = F.softmax(w, dim=-1)

# result is dot product of weights and values
output = w @ v
output.shape

torch.Size([4, 8, 16])

# Single attention head

In [22]:
class Head(nn.Module):

    def __init__(self, n_embed, head_size, block_size, dropout=0.1):
        super().__init__()
        self.normalize_factor = head_size**0.5
        self.key = nn.Linear(n_embed, head_size)
        self.query = nn.Linear(n_embed, head_size)
        self.value = nn.Linear(n_embed, head_size)
        self.register_buffer('tril', torch.tril(torch.ones((block_size, block_size))))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        w = q @ k.transpose(-2, -1) / self.normalize_factor
        w = w.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # using mask makes it decoder block, in encoder every token can communicate
        w = F.softmax(w, dim=-1)
        w = self.dropout(w)

        output = w @ v
        return output

# Adding Multihead Attention

In [23]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size, block_size, n_embed, dropout=0.1):
        super().__init__()
        self.heads = nn.ModuleList([Head(n_embed=n_embed, head_size=head_size, block_size=block_size, dropout=dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embed, n_embed)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

# Adding feedforward after multihead attention

In [24]:
class FeedForward(nn.Module):

    def __init__(self, n_embed, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

# Creating independent transformer block

In [25]:
class Block(nn.Module):

    def __init__(self, n_embed, n_head, block_size, dropout=0.1):
        super().__init__()
        head_size = n_embed // n_head
        self.sa = MultiHeadAttention(num_heads=n_head, head_size=head_size, n_embed=n_embed, block_size=block_size, dropout=dropout)
        self.ffwd = FeedForward(n_embed, dropout=dropout)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)
    def forward(self, x):
        x = x + self.sa(self.ln1(x)) # adding because of residual connection, better optimized
        x = x + self.ffwd(self.ln2(x))
        return x

In [31]:
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size, block_size, n_embed, n_layers, n_head, dropout, device):
        super().__init__()
        # each token reads directly off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.blocks = nn.Sequential(
            *[Block(n_embed, n_head=n_head, block_size=block_size, dropout=dropout) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)
        self.block_size = block_size
        self.device = device

    def forward(self, idx: torch.Tensor, targets=None):

        B, T = idx.shape

        # idx and targets are both (B, T) tensor of integers
        token_embeddings = self.token_embedding_table(idx) # (B, T, n_embed)
        position_embeddings = self.position_embedding_table(torch.arange(T).to(self.device)) # (T, n_embed)
        x = token_embeddings + position_embeddings
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.reshape(B*T, C)
            targets = targets.reshape(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        
        for _ in range(max_new_tokens):

            idx_cond = idx[:, -self.block_size:]
            # idx is (B,T) array of indices in the current context
            logits, loss = self(idx_cond)
            # focus only on last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)

        return idx

In [34]:
vocab_size = len(chars)
batch_size = 64
block_size = 256
max_iters = 5000
lr = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_embed = 192
n_head = 6
n_layer = 3
dropout = 0.2

model = BigramLanguageModel(vocab_size, block_size, n_embed, n_layers=4, n_head=4, dropout=dropout, device=device)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

In [35]:
for steps in range(max_iters):

    xb, yb = get_batch("train", batch_size)
    xb, yb = xb.to(device, dtype=torch.long), yb.to(device, dtype=torch.long)

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if not steps % 500:
        print(loss.item())

4.3835015296936035
2.3832976818084717
2.02719783782959
1.8189316987991333
1.7119241952896118
1.6209388971328735
1.558349847793579
1.5389270782470703
1.5106693506240845
1.4303717613220215


In [39]:
print(decode(model.generate(idx=torch.zeros((1,1), dtype=torch.long).to(device), max_new_tokens=1000)[0].tolist()))


DERBOLT:
Here do bell lay her and if thank you.

KING Lonce, I finight thine he, I make to thee.

GLOUCESTER:
But atte, for thy some to liberous theport,
Up, repens, as I come,
Your eye my barthen the promillcus of King,
With undo edting thou not hangst himsper Banister:
I'll good vill; ve's this he killandence; I live there
That to-dispine, enough our eyes, in ascent my ldness.
Sompeyon what enjoie, when with than the tooke! how is fair.

ROMEO:
'Tis not love arms in their,
Far live want to'er frach theme, mights he,
and mother, welcome-silver of me prink
Lives mother forus out eyes stockness.

VAULIA:
Nry!
With usures, them flower secondening become repots,
That my fault my lordshipp'ds, frone, think.

VIRGILANUS:
Might vey.

ISHOP OF CARISTRE:
As I do let teepance a Roman my your quistland
justers of marqure, my head duke father,
The touch as I uch are of sacreat up the more the some o'ersh.
What bad, ho must in the cryates homised.

BENVOLIOS:
'Tis the makes your own sword,
I shal